### Chaos Induction Engine

This notebook deliberately induces failure modes in the otherwise "Ideal" baseline data.
The goal is to introduce realism and messiness into the data as observed in Quick-commerce supply chain pipelines.

The failures are not random, each failure points to a physical cause.  

Read full fault specification: [`chaos_library.md`](chaos_library.md)



In [1]:
import yaml
import random
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta

with open("config.yaml")  as f:
    cfg = yaml.safe_load(f)

random.seed(cfg["simulation"]["random_seed"])
np.random.seed(cfg["simulation"]["random_seed"])

# baseline events
df_events = pd.read_csv("events_baseline.csv", parse_dates=["event_timestamp"])

## dimension tables are needed for targeted chaos
df_scanner= pd.read_csv("dim_scanner.csv")
df_product = pd.read_csv("dim_product.csv")
df_node = pd.read_csv("dim_node.csv")
df_supplier = pd.read_csv("dim_supplier.csv")

## pre-build targeting sets (used across multiple chaos methods)
dead_rtc_scanners = set(df_scanner[df_scanner["battery_backed_rtc"] == False]["device_id"])
degraded_scanners = set(df_scanner[df_scanner["is_degraded"] == True]["device_id"])
old_firmware_devices = set(df_scanner[df_scanner["firmware_version"] == "v1.1.0"]["device_id"])
slow_scanners = set(df_scanner[df_scanner["avg_sync_latency_ms"] > 500]["device_id"])
high_failure_devices = set(df_scanner[df_scanner["failure_count_30d"] > 5]["device_id"])

email_suppliers = set(df_supplier[df_supplier["edi_format"] == "email_manual"]["supplier_id"])
xml_suppliers = set(df_supplier[df_supplier["edi_format"] == "SFTP_XML"]["supplier_id"])
csv_suppliers = set(df_supplier[df_supplier["edi_format"] == "SFTP_CSV"]["supplier_id"])

email_products = set(df_product[df_product["supplier_id"].isin(email_suppliers)]["product_id"])
xml_products = set(df_product[df_product["supplier_id"].isin(xml_suppliers)]["product_id"])
csv_products = set(df_product[df_product["supplier_id"].isin(csv_suppliers)]["product_id"])

print(f"Baseline events loaded: {len(df_events):,}")
print(f"dead RTC scanners: {len(dead_rtc_scanners)}")
print(f"Degraded scanners: {len(degraded_scanners)}")
print(f"Old firmware devices: {len(old_firmware_devices)}")
print(f"email_manual products: {len(email_products)}")
print(f"SFTP_XML products: {len(xml_products)}")

Baseline events loaded: 365,635
dead RTC scanners: 47
Degraded scanners: 66
Old firmware devices: 103
email_manual products: 28
SFTP_XML products: 21


In [3]:
class SupplyChainChaosEngine:
    """ 
    Injects 25 physically attributed failure modes into the clean baseline.

    EVery chaos method targets specific rows based on scanner properties, supplier
    EDI format, or operational condidtions, never uniform sampling. 

    Target sets a pre-built in the cell above and passed in at construction. 
    Each method returns self for chaining. 

    Usage:
    engine = SupplyChainChaosEngine(df-events, df_node)
    (engine.apply_temporal_chaos().apply_structural_chaos
            .apply_operational_chaos().apply_market_chaos()
            .apply_lad_chaos().save())
    """

    def __init__(self, df_events, df_node):
        self.df = df_events.copy()
        self.df_nodes = df_node.copy()
        self.df["event_timestamp"] = pd.to_datetime(self.df["event_timestamp"])
        self.injection_log = [] # tracks what was injected for summary

    def _log(self, chaos_id, name, rows_affected):
        "Records each injection for post run summary"
        self.injection_log.append({
            "chaos_id": chaos_id,
            "name": name,
            "rows_affected": rows_affected,
        })
        print(f" [{chaos_id: 02d}] {name:<40} {rows_affected:>6} rows")
    def apply_temporal_chaos(self):
        raise NotImplementedError("implement in next cell")
    
    def apply_structural_chaos(self):
        raise NotImplementedError("implement in next cell")
    
    def apply_operational_chaos(self):
        raise NotImplementedError("implement in the next cell")
    
    def apply_market_chaos(self):
        raise NotImplementedError("implement in the next cell")
    def apply_lad_chaos(self):
        raise NotImplementedError("implement in next cell")
    def save(self, events_path = "events_chaotic.csv", 
             nodes_path = "nodes_chaotic.csv"):
        import os
        os.makedirs("data", exist_ok= True)
        self.df.to_csv(events_path, index = False)
        self.df_nodes.to_csv(nodes_path, index= False)
        print(f"\nSaved : {events_path} ({len(self.df):,} rows)")
        print(f"Saved: {nodes_path} ({len(self.df_nodes)} rows)")

    def summary(self):
        print("\n === Chaos Injection Summary ===")
        total = 0
        for entry in self.injection_log:
            print(f" [{entry['chaos_id']:02d}] {entry['name']:<40} {entry['rows_affected']:>6}")
            total += entry["rows_affected"]
        print(f"\n Total rows touched: {total:,}")
        print(f" Total events in chaotic dataset: {len(self.df):,}")

# confirm class instantiates cleanly
engine_test = SupplyChainChaosEngine(df_events, df_node)
print(f"ChaosEngine ready.")
print(f" Events loaded: {len(engine_test.df):,}")
print(f" Nodes loaded: {len(engine_test.df_nodes)}")

ChaosEngine ready.
 Events loaded: 365,635
 Nodes loaded: 120


### **Apply Temporal Chaos:**


In [4]:
def apply_temporal_chaos(self):
    """ 
    Injects 6 temporal failures.
    All target specific scanner properties from dim_scanner.
    See chaos_library.md for full specifications.
    """
    print("Applying temporal chaos...")

    # --1.1 UTC/IST Drift
    # scanners with dead rtc battery reset to utc on power loss.
    # all their events are 5h 30min behind real time
    mask = self.df["device_id"].isin(dead_rtc_scanners)
    affected = mask.sum()
    self.df.loc[mask, "event_timestamp"] -= pd.Timedelta(hours=5, minutes = 30)
    self._log(1, "UTC/IST Drift, affected")

    # --1.2 Timezone double-conversion
    # v1.1.0 firmware: utc to ist applied twice so 11h ahead
    #  applied to 2% of old firmware (different rows from drift) 

    old_fw_mask = self.df["device_id"].isin(old_firmware_devices)
    old_fw_idx = self.df[old_fw_mask & ~mask].sample(
        frac = 0.02, random_state = 42
    ).index
    self.df.loc[old_fw_idx, "event_timestamp"] += pd.Timedelta(hours=11)
    self._log(2, "Timezone Double-Conversion", len(old_fw_idx))

    # --1.3 Batch Heartbeat 
    # high latency scanners upload all the data buffered events at 23:59:59
    # pick 3 random dates, force all slow scanner events to EOD
    batch_dates = pd.to_datetime(
        pd.Series(self.df["event_timestamp"].dt.unique())
        .sample(3, random_state=42).values
    )
    for batch_date in batch_dates:
        bh_mask = (
            self.df["device_id"].isin(slow_scanners) & 
            (self.df["event_timestamp"].dt.date == batch_date.date())
        )
        self.df.loc[bh_mask, "event_timestamp"] = pd.Timestamp(
            f"{batch_date.date()} 23:59:59"
        )
    total_bh = self.df["device_id"].isin(slow_scanners).sum()
    self._log(3, "Batch Heartbeat (3 dates)", total_bh)

    # --1.4 Future Timestamp
    # v1.1.0 firmware clock drifts forward after battery replacement
    # 2% of old firmware rows shifted 2-45 days into the future
    future_idx = self.df[old_fw_mask].sample(frac= 0.02, random_state=43).index
    self.df.loc[future_idx, "event_timestamp"] = (
        self.df.loc[future_idx, "event_timestamp"]
        + pd.to_timedelta(
            [random.randint(2, 45) for _ in range(len(future_idx))],
            unit="D"
        )
    )
    self._log(4, "Future Timestamp", len(future_idx))

    # --1.5 Midnight Rollover
    # old ERP systems truncate datetime to date only during export
    # 5% of v1.1.0 rows get time component zeroed    
    midnight_idx = self.df[old_fw_mask].sample(frac=0.05, random_state=44).index
    self.df.loc[midnight_idx, "event_timestamp"] = (
        self.df.loc[midnight_idx, "event_timestamp"].dt.normalize()
    )
    self._log(5, "Midnight Rollover", len(midnight_idx))    


    # --1.6 Processing Lag Smear
    ## degraded scanners accumulate massive processing delays
    # 2% of degraded scanner rows get lag set to 155.5 hours
    lag_idx = self.df[
        self.df["device_id"].isin(degraded_scanners)
    ].sample(frac=0.02, random_state=45).index

    self.df.loc[lag_idx, "processing_lag_hours"] = 155.5
    self._log(6, "Processing Lag Smear", len(lag_idx))

    return self   

# patch the method on the class
SupplyChainChaosEngine.apply_temporal_chaos = apply_temporal_chaos
print("apply_temporal_chaos() patched onto ChaosEngine.")

apply_temporal_chaos() patched onto ChaosEngine.


### **Apply Structural Chaos:**